In [2]:
# ==========================================================
# Notebook 04: FAISS Vector Index
# ==========================================================

import faiss
import pickle
import numpy as np
import pandas as pd

from sentence_transformers import SentenceTransformer

d:\Books\projects-nlp-transformers-learning\.projectnlps\Lib\site-packages\sentence_transformers\cross_encoder\CrossEncoder.py:11: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm, trange


In [3]:
chunks_df = pd.read_csv("data/processed/document_chunks.csv")

chunks_df.head()

,chunk_id,source_document,page_number,chunk_text,chunk_length
0,1,annual_report.pdf,1,ABBOTT INDIA LIMITED ANNUAL REPORT 2024-25 TOG...,68
1,2,annual_report.pdf,2,Abbott India Limited CONTENTS Company Overview...,500
2,3,annual_report.pdf,2,of Directors 14 Independent Auditor’s Report 1...,494
3,4,annual_report.pdf,2,34 get and stay healthy.” Munir Shaikh Wellnes...,348
4,5,annual_report.pdf,2,. Wellness for Environment 54 We cannot guaran...,470


In [5]:
with open("data/embeddings/chunk_embeddings.pkl", "rb") as file:

    chunk_embeddings = pickle.load(file)

chunk_embeddings = np.array(chunk_embeddings).astype("float32")

print(chunk_embeddings.shape)

chunk_embeddings

(1804, 384)


array([[ 0.02380635,  0.0235075 , -0.02489271, ..., -0.12640804,
        -0.04217105, -0.01912927],
       [ 0.02019244, -0.03724829, -0.04387859, ..., -0.08506396,
         0.03206871, -0.02699326],
       [ 0.03979272,  0.0589406 , -0.0126948 , ..., -0.04564386,
         0.00386457, -0.07151814],
       ...,
       [-0.02146751, -0.05283569, -0.04981942, ..., -0.11791276,
         0.03287147,  0.01242815],
       [-0.04618213,  0.00223527, -0.03681865, ..., -0.10148104,
        -0.03554746, -0.00867944],
       [-0.06429781, -0.02824027,  0.04525841, ...,  0.00623695,
         0.01682922, -0.03065148]], dtype=float32)

In [6]:
faiss.normalize_L2(chunk_embeddings)

In [9]:
embedding_dimension = chunk_embeddings.shape[1]

print("Embedding Dimension:", embedding_dimension)

Embedding Dimension: 384


In [10]:
faiss_index = faiss.IndexFlatIP(embedding_dimension)

In [11]:
faiss_index

<faiss.swigfaiss_avx2.IndexFlatIP; proxy of <Swig Object of type 'faiss::IndexFlatIP *' at 0x000001FCDFA840F0> >

In [12]:
faiss_index.add(chunk_embeddings)

print("Total Indexed Vectors:", faiss_index.ntotal)

Total Indexed Vectors: 1804


In [13]:
faiss_index

<faiss.swigfaiss_avx2.IndexFlatIP; proxy of <Swig Object of type 'faiss::IndexFlatIP *' at 0x000001FCDFA840F0> >

In [14]:
embedding_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

query = "What was the company's revenue?"

query_embedding = embedding_model.encode(query)

d:\Books\projects-nlp-transformers-learning\.projectnlps\Lib\site-packages\huggingface_hub\file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


In [16]:
query_embedding.shape

(384,)

In [17]:
query_embedding = np.array([query_embedding]).astype("float32")

faiss.normalize_L2(query_embedding)

print(query_embedding.shape)

(1, 384)


In [18]:
top_k = 5

scores, indices = faiss_index.search(query_embedding, top_k)

In [19]:
scores, indices

(array([[0.56774926, 0.53758097, 0.5044949 , 0.50315535, 0.49847978]],
       dtype=float32),
 array([[1463, 1464,   11, 1460, 1423]], dtype=int64))

In [20]:
print("Similarity Scores:")

print(scores)

print("\nRetrieved Indices:")

print(indices)

Similarity Scores:
[[0.56774926 0.53758097 0.5044949  0.50315535 0.49847978]]

Retrieved Indices:
[[1463 1464   11 1460 1423]]


In [21]:
retrieved_chunks = chunks_df.iloc[indices[0]].copy()

retrieved_chunks["similarity_score"] = scores[0]

retrieved_chunks

,chunk_id,source_document,page_number,chunk_text,chunk_length,similarity_score
1463,1464,annual_report.pdf,102,. Opening Stock Finished goods 108.21 126.67 (...,495,0.567749
1464,1465,annual_report.pdf,102,- Sales Return (154.11) (168.31) Work-in-progr...,493,0.537581
11,12,annual_report.pdf,4,. $ 42 Billion WORLDWIDE SALES The Abbott we H...,355,0.504495
1460,1461,annual_report.pdf,102,and carriers claims 0.67 2.82 Outside India (G...,494,0.503155
1423,1424,annual_report.pdf,99,. Current income tax : 2. Capital Reserve Curr...,274,0.498480


In [22]:
def retrieve_documents(query, embedding_model, faiss_index, chunk_dataframe, top_k=5):

    query_embedding = embedding_model.encode(query)

    query_embedding = np.array([query_embedding]).astype("float32")

    faiss.normalize_L2(query_embedding)

    scores, indices = faiss_index.search(query_embedding, top_k)

    results = chunk_dataframe.iloc[indices[0]].copy()

    results["similarity_score"] = scores[0]

    return results

In [23]:
results = retrieve_documents(
    query="What was the company's revenue?",
    embedding_model=embedding_model,
    faiss_index=faiss_index,
    chunk_dataframe=chunks_df,
    top_k=3,
)

results[["page_number", "similarity_score", "chunk_text"]]

,page_number,similarity_score,chunk_text
1463,102,0.567749,. Opening Stock Finished goods 108.21 126.67 (...
1464,102,0.537581,- Sales Return (154.11) (168.31) Work-in-progr...
11,4,0.504495,. $ 42 Billion WORLDWIDE SALES The Abbott we H...


In [24]:
FAISS_PATH = "data/embeddings/" "faiss_index.bin"

faiss.write_index(faiss_index, FAISS_PATH)

print("FAISS index saved.")

FAISS index saved.


In [25]:
loaded_index = faiss.read_index(FAISS_PATH)

print("Indexed Vectors:", loaded_index.ntotal)

Indexed Vectors: 1804
